In [1]:
# ============================================================
# dask_pipeline.ipynb
# Dask ML Pipeline - NYC Taxi Dataset
# ============================================================

# 1. Imports

import time

import requests
import dask.dataframe as dd
from pydantic import ByteSize
from dask.distributed import Client
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBRegressor

In [2]:
# 2. Start Dask client

client = Client()

client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 8
Total threads: 32,Total memory: 62.53 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35009,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38667,Total threads: 4
Dashboard: http://127.0.0.1:44541/status,Memory: 7.82 GiB
Nanny: tcp://127.0.0.1:35687,


In [3]:
# 3. Configuration

RUN_MODE = "local"

if RUN_MODE == "local":
    DATA_PATH = "../data/raw/taxi/yellow_tripdata_2026-01.parquet"
else:
    DATA_PATH = "gs://your-bucket/taxi/yellow_tripdata_2026-01.parquet"

LOOKUP_PATH = "../data/raw/taxi/taxi_zone_lookup.csv"

SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"

TARGET = "fare_amount"

FEATURES = [
    "trip_distance",
    "passenger_count",
    "PULocationID",
    "DOLocationID",
    "payment_type",
]

In [4]:
from collections.abc import Generator
from contextlib import contextmanager
from datetime import timedelta
from pathlib import Path
from time import perf_counter
from typing import TypedDict

class Duration(TypedDict):
    duration: timedelta


def _mem_str(path: str) -> str:
    path = Path(path)
    size = path.stat().st_size
    return ByteSize(size).human_readable(decimal=True)

@contextmanager
def timeit(name: str) -> Generator[dict]:
    result = {"duration": timedelta()}
    start_time = perf_counter()
    yield result
    end_time = perf_counter()
    duration = timedelta(seconds=end_time - start_time)
    print(f"{name} took {duration}")
    result["duration"] = duration

In [5]:
from datetime import timedelta

# 4. Load data
df: dd.DataFrame = None
try:
    with timeit("Loading data"):
        df = dd.read_parquet(DATA_PATH)
except FileNotFoundError:
    response = requests.get(SOURCE_URL)
    response.raise_for_status()
    Path(DATA_PATH).parent.mkdir(parents=True, exist_ok=True)
    with open(DATA_PATH, "wb") as f:
        f.write(response.content)
    print(f"Downloaded {DATA_PATH}, {_mem_str(DATA_PATH)}")


if df is None:
    with timeit("Loading data"):
        df = dd.read_parquet(DATA_PATH)

location_lookup = dd.read_csv("../data/raw/taxi/taxi_zone_lookup.csv")

df.head()

Loading data took 0:00:00.006930


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [6]:
# 5. Basic dataset inspection

print("Columns:")
print(df.columns)

print("Dtypes:")
print(df.dtypes)

print("Number of partitions:")
print(df.npartitions)

Columns:
Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='str')
Dtypes:
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                   int64
trip_distance                   float64
RatecodeID                        int64
store_and_fwd_flag               string
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount 

In [7]:
# 6. Preprocessing

df = df[FEATURES + [TARGET]]
df = df.dropna()
df = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["passenger_count"] > 0)
]
df.tail()

,trip_distance,passenger_count,PULocationID,DOLocationID,payment_type,fare_amount
2636822,2.75,2.0,166,237,1,17.0
2636827,1.86,1.0,230,239,1,12.1
2636828,2.66,1.0,239,262,1,16.3
2636829,0.36,4.0,170,170,1,5.8
2636830,1.15,1.0,230,229,1,10.7


In [8]:
def benchmark_dask(data_path, lookup_path):
    metrics = {}

    # 1. Read Data
    with timeit("1. Read Data") as duration:
        df = dd.read_parquet(data_path)
        df_lookup = dd.read_csv(lookup_path)
        df_lookup["LocationID"] = df_lookup["LocationID"].astype(df["PULocationID"].dtype)
        # Dask reads lazily; force compute on length to evaluate reading execution
        _ = len(df)
    metrics['1. Read Data'] = duration["duration"]

    # 2. Count Index (Simple count)
    with timeit("2. Count Operation") as duration:
        _ = df.count().compute()
    metrics['2. Count Operation'] = duration["duration"]

    # 3. Complex Arithmetic Formula
    with timeit("3. Complex Arithmetic") as duration:
        arithmetic_res = (df['fare_amount'] + df['tip_amount']) * 1.5 / (df['passenger_count'] + 1)
        _ = arithmetic_res.compute()
    metrics['3. Complex Arithmetic'] = duration["duration"]

    # 4. Statistical Standard Deviation
    with timeit("4. Standard Deviation") as duration:
        _ = df['trip_distance'].std().compute()
    metrics['4. Standard Deviation'] = duration["duration"]

    # 5. GroupBy Aggregation
    with timeit("5. GroupBy Mean") as duration:
        _ = df.groupby('passenger_count')['fare_amount'].mean().compute()
    metrics['5. GroupBy Mean'] = duration["duration"]

    # 6. Merge/Join with Count
    with timeit("6. Join & Count") as duration:
        joined = dd.merge(df, df_lookup, left_on='PULocationID', right_on='LocationID', how='inner')
        _ = len(joined)
    metrics['6. Join & Count'] = duration["duration"]

    return metrics

In [9]:
benchmarks = benchmark_dask(DATA_PATH, LOOKUP_PATH)

1. Read Data took 0:00:00.015297
2. Count Operation took 0:00:00.501338
3. Complex Arithmetic took 0:00:00.174486
4. Standard Deviation took 0:00:00.085644
5. GroupBy Mean took 0:00:00.116355
6. Join & Count took 0:00:00.494136


In [10]:
from pathlib import Path

yellow2025: dd.DataFrame = None
for taxi_file in Path("../data/raw/taxi").glob("yellow*.parquet"):
    yellow = dd.read_parquet(taxi_file)
    yellow2025 = dd.concat([yellow2025, yellow]) if yellow2025 is not None else yellow

yellow2025.to_parquet("../data/raw/taxi/2025_yellow_tripdata.parquet", write_index=False)

In [11]:
benchmarks = benchmark_dask("../data/raw/taxi/2025_yellow_tripdata.parquet", LOOKUP_PATH)

1. Read Data took 0:00:00.018700
2. Count Operation took 0:00:01.377925
3. Complex Arithmetic took 0:00:03.061665
4. Standard Deviation took 0:00:00.463904
5. GroupBy Mean took 0:00:00.490679
6. Join & Count took 0:00:00.657981
